# Question 5: SVM Performance with Class Imbalance

**Objective:** Analyze SVM performance under class imbalance and demonstrate how class weights improve performance.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

## Create Imbalanced Dataset

In [ ]:
# Generate synthetic imbalanced dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1 (imbalanced)
    flip_y=0.01,
    random_state=42
)

print("IMBALANCED DATASET INFORMATION")
print("="*60)
print(f"Total samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"\nClass distribution:")
unique, counts = np.unique(y, return_counts=True)
for cls, count in zip(unique, counts):
    percentage = (count / len(y)) * 100
    print(f"  Class {cls}: {count} samples ({percentage:.1f}%)")

print(f"\nImbalance Ratio: {counts[0]/counts[1]:.2f}:1")

## Visualize Class Imbalance

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
class_counts = Counter(y)
axes[0].bar(['Majority (0)', 'Minority (1)'], 
           [class_counts[0], class_counts[1]], 
           color=['#4ECDC4', '#FF6B6B'])
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

for i, (cls, count) in enumerate(class_counts.items()):
    axes[0].text(i, count, str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
colors = ['#4ECDC4', '#FF6B6B']
axes[1].pie([class_counts[0], class_counts[1]], 
           labels=['Majority (0)', 'Minority (1)'],
           autopct='%1.1f%%',
           colors=colors,
           startangle=90)
axes[1].set_title('Class Distribution Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Data Preprocessing

In [ ]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:")
train_counts = Counter(y_train)
for cls, count in train_counts.items():
    print(f"  Class {cls}: {count} samples ({count/len(y_train)*100:.1f}%)")

print(f"\nTest set class distribution:")
test_counts = Counter(y_test)
for cls, count in test_counts.items():
    print(f"  Class {cls}: {count} samples ({count/len(y_test)*100:.1f}%)")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model 1: SVM without Class Weights (Baseline)

In [ ]:
# Train SVM without class weights
svm_baseline = SVC(kernel='rbf', random_state=42)
svm_baseline.fit(X_train_scaled, y_train)

# Predictions
y_pred_baseline = svm_baseline.predict(X_test_scaled)

# Evaluation
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
precision_baseline = precision_score(y_test, y_pred_baseline)
recall_baseline = recall_score(y_test, y_pred_baseline)
f1_baseline = f1_score(y_test, y_pred_baseline)

print("SVM WITHOUT CLASS WEIGHTS (Baseline)")
print("="*60)
print(f"Accuracy:  {accuracy_baseline:.4f}")
print(f"Precision: {precision_baseline:.4f}")
print(f"Recall:    {recall_baseline:.4f}")
print(f"F1-Score:  {f1_baseline:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=['Majority (0)', 'Minority (1)']))

## Model 2: SVM with Balanced Class Weights

In [ ]:
# Train SVM with balanced class weights
svm_balanced = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_balanced.fit(X_train_scaled, y_train)

# Predictions
y_pred_balanced = svm_balanced.predict(X_test_scaled)

# Evaluation
accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
f1_balanced = f1_score(y_test, y_pred_balanced)

print("SVM WITH BALANCED CLASS WEIGHTS")
print("="*60)
print(f"Accuracy:  {accuracy_balanced:.4f}")
print(f"Precision: {precision_balanced:.4f}")
print(f"Recall:    {recall_balanced:.4f}")
print(f"F1-Score:  {f1_balanced:.4f}")

# Calculate class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print(f"\nCalculated Class Weights:")
print(f"  Class 0 (Majority): {class_weights[0]:.4f}")
print(f"  Class 1 (Minority): {class_weights[1]:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced, target_names=['Majority (0)', 'Minority (1)']))

## Model 3: SVM with Custom Class Weights

In [ ]:
# Train SVM with custom class weights (emphasize minority class even more)
custom_weights = {0: 1, 1: 15}  # Give more weight to minority class
svm_custom = SVC(kernel='rbf', class_weight=custom_weights, random_state=42)
svm_custom.fit(X_train_scaled, y_train)

# Predictions
y_pred_custom = svm_custom.predict(X_test_scaled)

# Evaluation
accuracy_custom = accuracy_score(y_test, y_pred_custom)
precision_custom = precision_score(y_test, y_pred_custom)
recall_custom = recall_score(y_test, y_pred_custom)
f1_custom = f1_score(y_test, y_pred_custom)

print("SVM WITH CUSTOM CLASS WEIGHTS {0:1, 1:15}")
print("="*60)
print(f"Accuracy:  {accuracy_custom:.4f}")
print(f"Precision: {precision_custom:.4f}")
print(f"Recall:    {recall_custom:.4f}")
print(f"F1-Score:  {f1_custom:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_custom, target_names=['Majority (0)', 'Minority (1)']))

## Performance Comparison

In [ ]:
# Compare all models
models = ['Baseline\n(No weights)', 'Balanced\nWeights', 'Custom\nWeights (15:1)']
metrics_data = {
    'Accuracy': [accuracy_baseline, accuracy_balanced, accuracy_custom],
    'Precision': [precision_baseline, precision_balanced, precision_custom],
    'Recall': [recall_baseline, recall_balanced, recall_custom],
    'F1-Score': [f1_baseline, f1_balanced, f1_custom]
}

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(models))
width = 0.2
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for i, (metric, values) in enumerate(metrics_data.items()):
    offset = width * (i - 1.5)
    bars = ax.bar(x + offset, values, width, label=metric, color=colors[i])
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('SVM Configuration', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Performance Comparison: Effect of Class Weights on Imbalanced Data', 
            fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

## Confusion Matrices Comparison

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

predictions = [y_pred_baseline, y_pred_balanced, y_pred_custom]
titles = ['Baseline (No Weights)', 'Balanced Weights', 'Custom Weights (15:1)']
cmaps = ['Reds', 'Blues', 'Greens']

for idx, (y_pred, title, cmap) in enumerate(zip(predictions, titles, cmaps)):
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes[idx],
               xticklabels=['Majority (0)', 'Minority (1)'],
               yticklabels=['Majority (0)', 'Minority (1)'])
    
    axes[idx].set_title(title, fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('True Label', fontsize=11)
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    
    # Calculate and display metrics
    tn, fp, fn, tp = cm.ravel()
    axes[idx].text(0.5, -0.15, 
                  f'TN={tn}, FP={fp}\nFN={fn}, TP={tp}',
                  transform=axes[idx].transAxes,
                  ha='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('Confusion Matrices: Impact of Class Weights', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- Baseline: High accuracy but poor minority class recall")
print("- Balanced: Better minority class detection")
print("- Custom: Highest minority class recall at cost of majority precision")

## ROC Curves Comparison

In [ ]:
# Calculate ROC curves
decision_baseline = svm_baseline.decision_function(X_test_scaled)
decision_balanced = svm_balanced.decision_function(X_test_scaled)
decision_custom = svm_custom.decision_function(X_test_scaled)

fpr_baseline, tpr_baseline, _ = roc_curve(y_test, decision_baseline)
fpr_balanced, tpr_balanced, _ = roc_curve(y_test, decision_balanced)
fpr_custom, tpr_custom, _ = roc_curve(y_test, decision_custom)

auc_baseline = roc_auc_score(y_test, decision_baseline)
auc_balanced = roc_auc_score(y_test, decision_balanced)
auc_custom = roc_auc_score(y_test, decision_custom)

# Plot ROC curves
plt.figure(figsize=(10, 8))
plt.plot(fpr_baseline, tpr_baseline, lw=2, label=f'Baseline (AUC = {auc_baseline:.4f})', color='#FF6B6B')
plt.plot(fpr_balanced, tpr_balanced, lw=2, label=f'Balanced Weights (AUC = {auc_balanced:.4f})', color='#4ECDC4')
plt.plot(fpr_custom, tpr_custom, lw=2, label=f'Custom Weights (AUC = {auc_custom:.4f})', color='#45B7D1')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves: Effect of Class Weights', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Precision-Recall Curves

In [ ]:
# Calculate Precision-Recall curves
precision_baseline, recall_baseline_curve, _ = precision_recall_curve(y_test, decision_baseline)
precision_balanced, recall_balanced_curve, _ = precision_recall_curve(y_test, decision_balanced)
precision_custom, recall_custom_curve, _ = precision_recall_curve(y_test, decision_custom)

# Plot Precision-Recall curves
plt.figure(figsize=(10, 8))
plt.plot(recall_baseline_curve, precision_baseline, lw=2, 
        label=f'Baseline (F1 = {f1_baseline:.4f})', color='#FF6B6B')
plt.plot(recall_balanced_curve, precision_balanced, lw=2, 
        label=f'Balanced Weights (F1 = {f1_balanced:.4f})', color='#4ECDC4')
plt.plot(recall_custom_curve, precision_custom, lw=2, 
        label=f'Custom Weights (F1 = {f1_custom:.4f})', color='#45B7D1')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves: Effect of Class Weights', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Demonstration on Real-World Dataset: Credit Card Fraud

In [ ]:
# Create another imbalanced dataset (simulating credit card fraud)
X_fraud, y_fraud = make_classification(
    n_samples=5000,
    n_features=30,
    n_informative=25,
    n_redundant=5,
    n_classes=2,
    weights=[0.98, 0.02],  # 98% legitimate, 2% fraud
    flip_y=0.01,
    random_state=42
)

print("\n" + "="*60)
print("CREDIT CARD FRAUD DATASET (Simulated)")
print("="*60)
print(f"Total transactions: {X_fraud.shape[0]}")
print(f"Number of features: {X_fraud.shape[1]}")
print(f"\nClass distribution:")
fraud_counts = Counter(y_fraud)
for cls, count in fraud_counts.items():
    cls_name = 'Legitimate' if cls == 0 else 'Fraud'
    percentage = (count / len(y_fraud)) * 100
    print(f"  {cls_name}: {count} ({percentage:.2f}%)")

print(f"\nImbalance Ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1")

In [ ]:
# Split and scale fraud dataset
X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(
    X_fraud, y_fraud, test_size=0.3, random_state=42, stratify=y_fraud
)

scaler_fraud = StandardScaler()
X_train_fraud_scaled = scaler_fraud.fit_transform(X_train_fraud)
X_test_fraud_scaled = scaler_fraud.transform(X_test_fraud)

# Train models on fraud dataset
print("\nTraining SVM models on fraud detection dataset...")
print("-" * 60)

# Without class weights
svm_fraud_baseline = SVC(kernel='rbf', random_state=42)
svm_fraud_baseline.fit(X_train_fraud_scaled, y_train_fraud)
y_pred_fraud_baseline = svm_fraud_baseline.predict(X_test_fraud_scaled)

# With balanced class weights
svm_fraud_balanced = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_fraud_balanced.fit(X_train_fraud_scaled, y_train_fraud)
y_pred_fraud_balanced = svm_fraud_balanced.predict(X_test_fraud_scaled)

print("\nBaseline (No Class Weights):")
print(classification_report(y_test_fraud, y_pred_fraud_baseline, 
                           target_names=['Legitimate', 'Fraud']))

print("\nWith Balanced Class Weights:")
print(classification_report(y_test_fraud, y_pred_fraud_balanced, 
                           target_names=['Legitimate', 'Fraud']))

In [ ]:
# Compare fraud detection performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrices
cm_fraud_baseline = confusion_matrix(y_test_fraud, y_pred_fraud_baseline)
cm_fraud_balanced = confusion_matrix(y_test_fraud, y_pred_fraud_balanced)

sns.heatmap(cm_fraud_baseline, annot=True, fmt='d', cmap='Reds', ax=axes[0],
           xticklabels=['Legitimate', 'Fraud'],
           yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title('Baseline (No Class Weights)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_fraud_balanced, annot=True, fmt='d', cmap='Blues', ax=axes[1],
           xticklabels=['Legitimate', 'Fraud'],
           yticklabels=['Legitimate', 'Fraud'])
axes[1].set_title('With Balanced Class Weights', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.suptitle('Fraud Detection: Confusion Matrices Comparison', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Calculate fraud detection metrics
tn_base, fp_base, fn_base, tp_base = cm_fraud_baseline.ravel()
tn_bal, fp_bal, fn_bal, tp_bal = cm_fraud_balanced.ravel()

print("\n" + "="*60)
print("FRAUD DETECTION ANALYSIS")
print("="*60)
print(f"\nBaseline Model:")
print(f"  Frauds Detected: {tp_base}/{tp_base + fn_base} ({tp_base/(tp_base + fn_base)*100:.1f}%)")
print(f"  Frauds Missed: {fn_base}")
print(f"  False Alarms: {fp_base}")

print(f"\nBalanced Model:")
print(f"  Frauds Detected: {tp_bal}/{tp_bal + fn_bal} ({tp_bal/(tp_bal + fn_bal)*100:.1f}%)")
print(f"  Frauds Missed: {fn_bal}")
print(f"  False Alarms: {fp_bal}")

print(f"\n✓ Improvement: {tp_bal - tp_base} more frauds detected!")
print(f"⚠ Trade-off: {fp_bal - fp_base} additional false alarms")

## Summary Table

In [ ]:
# Create comprehensive summary
summary_data = [
    {
        'Configuration': 'Baseline (No Weights)',
        'Dataset': 'Synthetic',
        'Accuracy': f"{accuracy_baseline:.4f}",
        'Precision': f"{precision_baseline:.4f}",
        'Recall': f"{recall_baseline:.4f}",
        'F1-Score': f"{f1_baseline:.4f}"
    },
    {
        'Configuration': 'Balanced Weights',
        'Dataset': 'Synthetic',
        'Accuracy': f"{accuracy_balanced:.4f}",
        'Precision': f"{precision_balanced:.4f}",
        'Recall': f"{recall_balanced:.4f}",
        'F1-Score': f"{f1_balanced:.4f}"
    },
    {
        'Configuration': 'Custom Weights (15:1)',
        'Dataset': 'Synthetic',
        'Accuracy': f"{accuracy_custom:.4f}",
        'Precision': f"{precision_custom:.4f}",
        'Recall': f"{recall_custom:.4f}",
        'F1-Score': f"{f1_custom:.4f}"
    },
    {
        'Configuration': 'Baseline (No Weights)',
        'Dataset': 'Fraud',
        'Accuracy': f"{accuracy_score(y_test_fraud, y_pred_fraud_baseline):.4f}",
        'Precision': f"{precision_score(y_test_fraud, y_pred_fraud_baseline):.4f}",
        'Recall': f"{recall_score(y_test_fraud, y_pred_fraud_baseline):.4f}",
        'F1-Score': f"{f1_score(y_test_fraud, y_pred_fraud_baseline):.4f}"
    },
    {
        'Configuration': 'Balanced Weights',
        'Dataset': 'Fraud',
        'Accuracy': f"{accuracy_score(y_test_fraud, y_pred_fraud_balanced):.4f}",
        'Precision': f"{precision_score(y_test_fraud, y_pred_fraud_balanced):.4f}",
        'Recall': f"{recall_score(y_test_fraud, y_pred_fraud_balanced):.4f}",
        'F1-Score': f"{f1_score(y_test_fraud, y_pred_fraud_balanced):.4f}"
    }
]

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*90)
print("COMPREHENSIVE SUMMARY - SVM WITH CLASS IMBALANCE")
print("="*90)
print(summary_df.to_string(index=False))
print("="*90)

## Conclusion

### Key Findings:

#### 1. **Impact of Class Imbalance on SVM:**
   - **Without class weights:** SVM tends to be biased toward the majority class
   - High overall accuracy can be misleading
   - Poor recall for minority class (many false negatives)
   - Model essentially ignores the minority class to maximize accuracy

#### 2. **Benefits of Class Weights:**
   - **Balanced weights:** Automatically computed as n_samples / (n_classes * n_samples_per_class)
   - Significantly improves minority class recall
   - Better F1-score indicating balanced performance
   - More suitable for imbalanced problems

#### 3. **Custom Class Weights:**
   - Allow fine-tuning based on problem requirements
   - Higher weights for minority class increase its detection
   - Trade-off between precision and recall
   - Useful when cost of missing minority class is high

#### 4. **Metrics Interpretation:**
   - **Accuracy:** Can be misleading with imbalanced data
   - **Precision:** How many predicted positives are actually positive
   - **Recall:** How many actual positives are correctly identified
   - **F1-Score:** Harmonic mean, better for imbalanced datasets
   - **AUC-ROC:** Overall discrimination ability across thresholds

#### 5. **Real-World Application (Fraud Detection):**
   - Baseline misses many fraud cases (high false negatives)
   - Balanced weights detect significantly more frauds
   - Small increase in false alarms is acceptable trade-off
   - Cost of missing fraud > cost of false alarm

### Recommendations:

1. **Always use class weights** when dealing with imbalanced data
2. **Start with 'balanced'** and adjust if needed
3. **Consider domain requirements:**
   - Medical diagnosis: Prioritize recall (don't miss diseases)
   - Spam detection: Balance precision and recall
   - Fraud detection: Prioritize recall (catch frauds)

4. **Evaluation metrics:**
   - Don't rely solely on accuracy
   - Use F1-score, precision, recall
   - Consider confusion matrix analysis
   - Plot ROC and Precision-Recall curves

5. **Alternative approaches:**
   - SMOTE (Synthetic Minority Over-sampling)
   - Random under-sampling of majority class
   - Ensemble methods (Random Forest, XGBoost)
   - Anomaly detection for extreme imbalance

### Key Takeaway:
**Class weights are essential for SVM performance on imbalanced datasets. They enable the model to give appropriate importance to minority classes, dramatically improving recall and F1-score at the cost of slightly reduced overall accuracy and precision. The choice of weights should align with the specific costs and requirements of your application.**